In [3]:
# Cell 1 — Setup: load ATT&CK catalogue + LM Studio connectivity check

import json, re, requests

# ── 1. Load ATT&CK STIX ───────────────────────────────────────────
ATTACK_JSON = "OSRs/ATTACK/enterprise-attack-v16.1.json"
with open(ATTACK_JSON, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects = stix_bundle.get("objects", [])

# ── 2. Build technique catalogue ──────────────────────────────────
parent_map  = {}   # sub-tcode → parent tcode
revoked_map = {}   # old tcode → new tcode

def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

# Index all objects by id
stix_by_id = {o["id"]: o for o in objects}

# Build maps
for o in objects:
    if o.get("type") != "relationship":
        continue
    src = stix_by_id.get(o.get("source_ref",""))
    tgt = stix_by_id.get(o.get("target_ref",""))
    if not src or not tgt:
        continue
    tc_src = get_tcode(src)
    tc_tgt = get_tcode(tgt)
    rt = o.get("relationship_type","")
    if rt == "subtechnique-of" and tc_src and tc_tgt:
        parent_map[tc_src] = tc_tgt
    elif rt == "revoked-by" and tc_src and tc_tgt:
        revoked_map[tc_src] = tc_tgt

def resolve_to_parent(tc):
    tc = revoked_map.get(tc, tc)
    tc = parent_map.get(tc, tc)
    tc = revoked_map.get(tc, tc)
    return tc

# Active parent techniques only
revoked_tcodes = set(revoked_map.keys())
for o in objects:
    if o.get("type") == "attack-pattern":
        if o.get("x_mitre_revoked") or o.get("revoked"):
            tc = get_tcode(o)
            if tc:
                revoked_tcodes.add(tc)

technique_ids   = []   # list of T-codes
technique_names = {}   # T-code → name
technique_descs = {}   # T-code → description

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    if o.get("x_mitre_revoked") or o.get("revoked"):
        continue
    tc = get_tcode(o)
    if not tc or "." in tc:   # skip sub-techniques
        continue
    if tc in revoked_tcodes:
        continue
    technique_ids.append(tc)
    technique_names[tc] = o.get("name", "")
    technique_descs[tc] = o.get("description", "")

technique_ids = sorted(set(technique_ids))
technique_id_set = set(technique_ids)

print(f"Active parent techniques: {len(technique_ids)}")
print(f"Sample: {technique_ids[:5]}")

# ── 3. Build technique list string for prompt ─────────────────────
technique_list_str = "\n".join(
    f"{tc}: {technique_names[tc]}"
    for tc in technique_ids
)
print(f"\nTechnique catalogue sample:")
print("\n".join(technique_list_str.split("\n")[:5]))
print("...")

# ── 4. LM Studio config ───────────────────────────────────────────
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LM_MODEL      = "qwen3.5-9b"   # label for logging only
TOP_K         = 5

SYSTEM_PROMPT = """You are a cybersecurity expert specializing in MITRE ATT&CK threat intelligence.

Your task: given a CVE vulnerability description, identify the most relevant ATT&CK techniques an adversary would use to exploit it.

Rules:
- You MUST only select techniques from the provided catalogue. Do not invent T-codes.
- Return EXACTLY a JSON object with one key "techniques" containing a list of T-codes, ranked by relevance (most relevant first).
- Return NO other text, explanation, or markdown. Only the raw JSON object.
- Select between 1 and 5 techniques. Only include techniques that are clearly relevant.

Output format (strictly follow this):
{"techniques": ["T1190", "T1059", "T1078"]}"""

def build_user_prompt(cve_description, db_hint=None):
    prompt = (f"CVE Description:\n{cve_description}\n\n"
              f"Valid ATT&CK Technique Catalogue (T-code: Name):\n"
              f"{technique_list_str}\n\n"
              f"Identify the top-{TOP_K} most relevant ATT&CK parent "
              f"techniques for this CVE.")
    if db_hint:
        prompt += (f"\n\nHint: A vulnerability database suggests these "
                   f"techniques may be relevant: {db_hint}\n"
                   f"Use this as a guide but apply your own judgment.")
    return prompt

def call_llm(cve_description, db_hint=None, retries=2):
    payload = {
        "model"      : LM_MODEL,
        "messages"   : [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(
                                    cve_description, db_hint)},
        ],
        "temperature": 0.0,
        "max_tokens" : 150,
        "stream"     : False,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for attempt in range(retries + 1):
        try:
            resp    = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"].strip()
            content = re.sub(r'^```json\s*|^```\s*|```$', '',
                             content, flags=re.MULTILINE).strip()
            parsed  = json.loads(content)
            raw_tcs = parsed.get("techniques", [])
            valid   = [tc.strip().upper() for tc in raw_tcs
                       if tc.strip().upper() in technique_id_set]
            return valid, content
        except requests.exceptions.ConnectionError:
            print("✗ Cannot connect — is LM Studio server running on port 1234?")
            return [], ""
        except json.JSONDecodeError:
            if attempt < retries:
                import time; time.sleep(1)
                continue
            found = re.findall(r'T\d{4}', content)
            valid = [tc.upper() for tc in found
                     if tc.upper() in technique_id_set]
            return valid[:TOP_K], content
        except Exception as e:
            print(f"  Error: {e}")
            return [], ""

# ── 5. Connectivity check ─────────────────────────────────────────
print("\nTesting LM Studio connection...")
test_tcs, test_raw = call_llm(
    "SQL injection vulnerability allows unauthenticated remote "
    "attackers to execute arbitrary SQL commands via user input.")
if test_tcs:
    print(f"✓ Connected.")
    print(f"  Predicted : {test_tcs}")
    print(f"  Raw output: {test_raw}")
else:
    print(f"✗ Failed. Raw: {test_raw}")
    print("  → Start LM Studio → Local Server → Load model → Start server")

Active parent techniques: 214
Sample: ['T1001', 'T1003', 'T1005', 'T1006', 'T1007']

Technique catalogue sample:
T1001: Data Obfuscation
T1003: OS Credential Dumping
T1005: Data from Local System
T1006: Direct Volume Access
T1007: System Service Discovery
...

Testing LM Studio connection...
✗ Failed. Raw: 
  → Start LM Studio → Local Server → Load model → Start server


In [5]:
# Cell 1b — Debug raw LM Studio response

import requests, json

payload = {
    "model"      : LM_MODEL,
    "messages"   : [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(
            "SQL injection vulnerability allows unauthenticated remote "
            "attackers to execute arbitrary SQL commands via user input.")},
    ],
    "temperature": 0.0,
    "max_tokens" : 150,
    "stream"     : False,
    "chat_template_kwargs": {"enable_thinking": False},
}

try:
    resp = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
    print(f"HTTP status  : {resp.status_code}")
    print(f"Raw response :\n{resp.text[:2000]}")
except Exception as e:
    print(f"Exception: {e}")

HTTP status  : 200
Raw response :
{
  "id": "chatcmpl-nqtn06v1jgc991gjfc2ilp",
  "object": "chat.completion",
  "created": 1775176396,
  "model": "qwen3.5-9b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "",
        "reasoning_content": "Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: A CVE description (\"SQL injection vulnerability allows unauthenticated remote attackers to execute arbitrary SQL commands via user input.\").\n    *   Task: Identify the most relevant MITRE ATT&CK techniques from the provided catalogue.\n    *   Constraints:\n        *   Return EXACTLY a JSON object with one key \"techniques\".\n        *   List of T-codes, ranked by relevance (most relevant first).\n        *   No other text, explanation, or markdown. Only raw JSON.\n        *   Select between 1 and 5 techniques.\n        *   Only include clearly relevant techniques from the provided catalogue.\n\n2",
        "tool_calls": [

In [15]:
# Cell 1c — Fix: disable Qwen3 thinking mode + increase token budget

def build_user_prompt(cve_description, db_hint=None):
    # /nothink at start disables Qwen3 chain-of-thought
    prompt = ("/nothink\n\n"
              f"CVE Description:\n{cve_description}\n\n"
              f"Valid ATT&CK Technique Catalogue (T-code: Name):\n"
              f"{technique_list_str}\n\n"
              f"Identify the top-{TOP_K} most relevant ATT&CK parent "
              f"techniques for this CVE.")
    if db_hint:
        prompt += (f"\n\nHint: A vulnerability database suggests these "
                   f"techniques may be relevant: {db_hint}\n"
                   f"Use this as a guide but apply your own judgment.")
    return prompt

def call_llm(cve_description, db_hint=None, retries=2):
    payload = {
        "model"      : LM_MODEL,
        "messages"   : [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(
                                    cve_description, db_hint)},
        ],
        "temperature": 0.0,
        "max_tokens" : 5120,    # enough for JSON even with some thinking
        "stream"     : False,
    }
    for attempt in range(retries + 1):
        try:
            resp    = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
            resp.raise_for_status()
            data    = resp.json()
            choice  = data["choices"][0]["message"]

            # Qwen3: actual answer may be in content or reasoning_content
            content = choice.get("reasoning_content", "").strip()
            if not content:
                # fallback: sometimes JSON ends up in reasoning_content
                content = choice.get("reasoning_content", "").strip()
                # extract last JSON block from reasoning
                matches = re.findall(r'\{[^{}]+\}', content)
                content = matches[-1] if matches else ""

            content = re.sub(r'^```json\s*|^```\s*|```$', '',
                             content, flags=re.MULTILINE).strip()

            parsed  = json.loads(content)
            raw_tcs = parsed.get("techniques", [])
            valid   = [tc.strip().upper() for tc in raw_tcs
                       if tc.strip().upper() in technique_id_set]
            return valid, content

        except requests.exceptions.ConnectionError:
            print("✗ Cannot connect — is LM Studio server running?")
            return [], ""
        except json.JSONDecodeError:
            if attempt < retries:
                import time; time.sleep(1)
                continue
            found = re.findall(r'T\d{4}', content)
            valid = [tc.upper() for tc in found
                     if tc.upper() in technique_id_set]
            return valid[:TOP_K], content
        except Exception as e:
            print(f"  Error: {e}")
            return [], ""

# ── Retest ────────────────────────────────────────────────────────
print("Retesting with /nothink fix...")
test_tcs, test_raw = call_llm(
    """Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1)
    JNDI features used in configuration, log messages, and parameters do not protect against attacker 
    controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log
    message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution
    is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0
    (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed.
    Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects. """
    )
print(f"Predicted : {test_tcs}")
print(f"Raw output: {test_raw}")

Retesting with /nothink fix...
Predicted : ['T1190', 'T1203', 'T1190', 'T1203', 'T1190']
Raw output: Thinking Process:

1.  **Analyze the Request:**
    *   **Task:** Identify the most relevant MITRE ATT&CK techniques (T-codes) for a given CVE description.
    *   **Input:** CVE Description (Apache Log4j2 RCE vulnerability).
    *   **Constraints:**
        *   Only use provided T-code catalogue.
        *   Return EXACTLY a JSON object with key "techniques".
        *   List must contain 1-5 T-codes, ranked by relevance.
        *   No markdown, no explanation, only raw JSON.
    *   **CVE:** Apache Log4j2 RCE (Log4Shell). This is a Remote Code Execution vulnerability in a widely used logging library.

2.  **Analyze the CVE Description:**
    *   **Vulnerability:** JNDI features allow arbitrary code execution from LDAP servers when message lookup substitution is enabled.
    *   **Impact:** An attacker who can control log messages or parameters can execute arbitrary code loaded from L

In [7]:
from tqdm.auto import tqdm

print("==================================================")
print("EVALUATING TRUE SMET ARCHITECTURE ON FULL DATASET")
print("==================================================")

hits_at_1_smet = 0
hits_at_5_smet = 0
hits_at_10_smet = 0
n_cves = len(cve_texts)

for i in tqdm(range(n_cves), desc="SMET LR Inference"):
    true_labels = set(cve_ground_truths[i])
    text = cve_texts[i]
    
    # Run the exact SMET prediction logic
    # We grab top 10 so we can calculate R@1, R@5, and R@10
    preds = map_cve_smet_way(text, top_k=10)
    
    # Extract just the T-codes
    top_10_preds = [p['t_code'] for p in preds]
    
    # Calculate Hits
    if len(top_10_preds) > 0 and top_10_preds[0] in true_labels: 
        hits_at_1_smet += 1
    if len(true_labels.intersection(set(top_10_preds[:5]))) > 0: 
        hits_at_5_smet += 1
    if len(true_labels.intersection(set(top_10_preds))) > 0: 
        hits_at_10_smet += 1

print("\n" + "-" * 50)
print("TRUE SMET ARCHITECTURE RESULTS")
print("-" * 50)
print(f"Hit Rate@1:  {(hits_at_1_smet / n_cves) * 100:.2f}%")
print(f"Hit Rate@5:  {(hits_at_5_smet / n_cves) * 100:.2f}%")
print(f"Hit Rate@10: {(hits_at_10_smet / n_cves) * 100:.2f}%")
print("-" * 50)

EVALUATING TRUE SMET ARCHITECTURE ON FULL DATASET


c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'cve_texts' is not defined